In [1]:
import duckdb, pandas as pd

con = duckdb.connect("../data/processed/recalls.duckdb", read_only=True)

single = con.execute("""
WITH per AS (
    SELECT ODINO,
           MIN(received_date)          AS received_date,
           ANY_VALUE(narrative)        AS narrative,
           COUNT(DISTINCT system_norm) AS n_systems,
           ANY_VALUE(system_norm)      AS system
    FROM complaints
    WHERE system_norm NOT IN ('UNKNOWN OR OTHER', 'OTHER', '0', '')
    GROUP BY ODINO
)
SELECT ODINO, received_date, narrative, system
FROM per
WHERE n_systems = 1 AND narrative IS NOT NULL AND LENGTH(narrative) >= 50
""").df()

print(f"{len(single):,} single-system complaints")
print(single["system"].value_counts().head(20).to_string())

526,414 single-system complaints
system
ENGINE                          87748
AIR BAGS                        72239
POWER TRAIN                     60823
ELECTRICAL SYSTEM               59039
STEERING                        45859
SERVICE BRAKES                  38107
FUEL/PROPULSION SYSTEM          30984
STRUCTURE                       30110
VISIBILITY/WIPER                20094
EXTERIOR LIGHTING               16444
SUSPENSION                      14389
VEHICLE SPEED CONTROL            7743
FORWARD COLLISION AVOIDANCE      7263
SEATS                            6376
SEAT BELTS                       5902
WHEELS                           5725
TIRES                            4686
BACK OVER PREVENTION             4220
LATCHES/LOCKS/LINKAGES           2945
ELECTRONIC STABILITY CONTROL     2452


In [3]:
TOP = single["system"].value_counts().head(15).index.tolist()
data = single[single["system"].isin(TOP)].copy()
data["received_date"] = pd.to_datetime(data["received_date"])

train_pool = data[data["received_date"] < "2023-01-01"]
val_pool   = data[(data["received_date"] >= "2023-01-01") & (data["received_date"] < "2024-01-01")]
test_pool  = data[data["received_date"] >= "2024-01-01"]

# Cap per class in train/val so rare systems aren't swamped; keep test at natural mix
train = train_pool.sample(frac=1, random_state=42).groupby("system").head(3000)
val = val_pool.sample(frac=1, random_state=42).groupby("system").head(400)
test = test_pool.sample(min(len(test_pool), 12000), random_state=42)

for n, d in [("train", train), ("val", val), ("test", test)]:
    print(f"{n}: {len(d):,} rows, {d['system'].nunique()} classes")
print("\nTest class mix:")
print(test["system"].value_counts().to_string())

train: 44,724 rows, 15 classes
val: 6,000 rows, 15 classes
test: 12,000 rows, 15 classes

Test class mix:
system
ENGINE                         3085
POWER TRAIN                    1819
ELECTRICAL SYSTEM              1491
STEERING                        924
SERVICE BRAKES                  914
FUEL/PROPULSION SYSTEM          797
STRUCTURE                       708
AIR BAGS                        559
VISIBILITY/WIPER                446
FORWARD COLLISION AVOIDANCE     332
EXTERIOR LIGHTING               266
SUSPENSION                      255
SEATS                           151
SEAT BELTS                      148
VEHICLE SPEED CONTROL           105


In [4]:
unknown = con.execute("""
SELECT ODINO,
       MIN(received_date)   AS received_date,
       ANY_VALUE(narrative) AS narrative
FROM complaints
GROUP BY ODINO
HAVING BOOL_AND(system_norm = 'UNKNOWN OR OTHER')
   AND LENGTH(ANY_VALUE(narrative)) >= 50
""").df()
con.close()

print(f"{len(unknown):,} complaints filed only as UNKNOWN OR OTHER")
print(unknown["narrative"].sample(3, random_state=1).str[:300].to_string())

52,846 complaints filed only as UNKNOWN OR OTHER
52165    STRUTS ON THE REAR DOOR ARE FAILING. THE DOOR ...
32496    THE CAR CONTINUOUSLY CUTS OFF THE BRAKE LIGHT ...
2510     I WAS PULLING OUT OF A SERVICE BAY AFTER AN OI...


In [5]:
cols = ["ODINO", "received_date", "narrative", "system"]
train[cols].to_parquet("../data/processed/text_train.parquet", index=False)
val[cols].to_parquet("../data/processed/text_val.parquet", index=False)
test[cols].to_parquet("../data/processed/text_test.parquet", index=False)
unknown.to_parquet("../data/processed/text_unknown.parquet", index=False)

import os
for f in ["text_train", "text_val", "text_test", "text_unknown"]:
    print(f, f"{os.path.getsize(f'../data/processed/{f}.parquet')/1e6:.1f} MB")

text_train 13.7 MB
text_val 1.8 MB
text_test 4.1 MB
text_unknown 16.5 MB


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
import time

t0 = time.time()
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=200_000, sublinear_tf=True)
X_tr = vec.fit_transform(train["narrative"])
X_te = vec.transform(test["narrative"])

clf = LogisticRegression(max_iter=2000, C=4.0)
clf.fit(X_tr, train["system"])
pred = clf.predict(X_te)

print(f"Accuracy: {accuracy_score(test['system'], pred):.3f}")
print(f"Macro F1: {f1_score(test['system'], pred, average='macro'):.3f}")
print(f"Time: {time.time()-t0:.0f}s\n")
print(classification_report(test["system"], pred, digits=3))

Accuracy: 0.838
Macro F1: 0.804
Time: 33s

                             precision    recall  f1-score   support

                   AIR BAGS      0.841     0.882     0.861       559
          ELECTRICAL SYSTEM      0.816     0.668     0.735      1491
                     ENGINE      0.901     0.888     0.895      3085
          EXTERIOR LIGHTING      0.878     0.921     0.899       266
FORWARD COLLISION AVOIDANCE      0.743     0.828     0.783       332
     FUEL/PROPULSION SYSTEM      0.849     0.818     0.833       797
                POWER TRAIN      0.884     0.819     0.850      1819
                 SEAT BELTS      0.876     0.905     0.890       148
                      SEATS      0.783     0.861     0.820       151
             SERVICE BRAKES      0.941     0.867     0.902       914
                   STEERING      0.932     0.920     0.926       924
                  STRUCTURE      0.718     0.801     0.757       708
                 SUSPENSION      0.613     0.906     0.731 

In [7]:
from sklearn.metrics import confusion_matrix
cm = pd.DataFrame(confusion_matrix(test["system"], pred, labels=TOP), index=TOP, columns=TOP)
np_cm = cm.values.copy()
import numpy as np
np.fill_diagonal(np_cm, 0)
pairs = [(TOP[i], TOP[j], np_cm[i, j]) for i in range(len(TOP)) for j in range(len(TOP)) if np_cm[i, j] > 0]
pairs.sort(key=lambda x: -x[2])
print("Most common confusions (true → predicted):")
for t, p_, n in pairs[:10]:
    print(f"  {t:35s} → {p_:35s} {n}")

Most common confusions (true → predicted):
  ELECTRICAL SYSTEM                   → STRUCTURE                           128
  POWER TRAIN                         → ENGINE                              125
  ENGINE                              → POWER TRAIN                         84
  ELECTRICAL SYSTEM                   → ENGINE                              79
  STRUCTURE                           → VISIBILITY/WIPER                    70
  ENGINE                              → ELECTRICAL SYSTEM                   64
  ELECTRICAL SYSTEM                   → POWER TRAIN                         64
  ENGINE                              → VEHICLE SPEED CONTROL               63
  POWER TRAIN                         → VEHICLE SPEED CONTROL               61
  POWER TRAIN                         → SUSPENSION                          57
